In [2]:
import torch

In [3]:
SYSTEM_PROMPT = """You are a SEARCH assistant with a Python REPL. You search documents - nothing else.

OUTPUT FORMAT: Your response must START with ```python - no preamble, no explanation, just code.

CONSTRAINT: Your training data is IRRELEVANT. You know NOTHING about this document.
- Answering without searching = WRONG
- Explaining instead of searching = WRONG
- Any text before your code block = WRONG

TOOLS:
- `context` - the document (already loaded, DO NOT redefine)
- `llm_query(question, context[start:end])` - ask sub-LLM about a chunk

WORKFLOW:
1. Write ```python with print() to search
2. STOP immediately after code block
3. Wait for output (appears in next message)
4. Search more OR give FINAL(answer)

SEARCH STRATEGY:
- If find() returns -1, try DIFFERENT keywords (not the same one)
- Try simpler terms, synonyms, related concepts
- Read surrounding text with context[idx:idx+500] to understand what you found

DO NOT:
- Explain what you're doing
- Answer from memory
- Write multiple code blocks
- Add text before ```python
- Redefine the context variable

When done searching, end with: FINAL(your evidence-based answer)"""

In [4]:
!python /workspace/generate_training_data.py

Loaded 155 total traces
RLM Training Data Generator v2

--- Phase 1: Loading documents ---
Loading Gutenberg texts...
  [cavalry_outpost] cached: cavalry_outpost.txt
  [household_physician] cached: household_physician.txt
  [boston_cooking_school] cached: boston_cooking_school.txt
  [rules_of_road_sea] cached: rules_of_road_sea.txt
  [cushings_manual] cached: cushings_manual.txt
Loading 8 synthetic document(s)...
  [product_spec] written (3,243 chars)
  [lease_agreement] written (4,533 chars)
  [lab_report] written (3,980 chars)
  [meeting_minutes] written (5,053 chars)
  [employee_handbook] written (5,766 chars)
  [safety_data_sheet] written (5,276 chars)
  [grant_proposal] written (5,866 chars)
  [it_incident_report] written (6,528 chars)
Loading GPO style manual...
  [gpo_manual] loaded (147,114 chars)
Total documents loaded: 14

Loaded 14 documents:
  cavalry_outpost: 64,392 chars
  household_physician: 181,100 chars
  boston_cooking_school: 129,641 chars
  rules_of_road_sea: 25,01

In [5]:
import json

with open("/workspace/data/rlm_training_v2.json") as f:
    dataset = json.load(f)

print(f"Total examples: {len(dataset)}")
print(f"First example has {len(dataset[0]['messages'])} messages")

for msg in dataset[0]["messages"]:
    role = msg["role"]
    content = msg["content"][:120] + "..." if len(msg["content"]) > 120 else msg["content"] 
    print(f"\n[{role}]: {content}")

Total examples: 155
First example has 4 messages

[user]: You are a SEARCH assistant with a Python REPL. You search documents - nothing else.

OUTPUT FORMAT: Your response must S...

[assistant]: I'll search for the keyword 'patrol' to find relevant passages.

```python
idx = context.find('patrol')
print(f'Found at...

[user]: Output:
Found at position: 7111
patrols to; whether he thinks that the

 Picquet has been well posted, at night as well ...

[assistant]: FINAL(The manual instructs that patrols should be sent out at irregular intervals along routes that cover the front and ...


In [6]:
from transformers import AutoTokenizer
import numpy as np
tokenizer = AutoTokenizer.from_pretrained("unsloth/DeepSeek-R1-Distill-Qwen-14B-unsloth-bnb-4bit") #loads the tokenizer
lengths = []
for example in dataset:
    lengths.append(len(tokenizer.apply_chat_template(example["messages"], tokenize = True)))
data = np.array(lengths)
np.max(data)

np.int64(822)

In [7]:
import unsloth
from unsloth import FastLanguageModel
max_seq_length = 1024
model, tokenizer = FastLanguageModel.from_pretrained("unsloth/DeepSeek-R1-Distill-Qwen-14B-unsloth-bnb-4bit", max_seq_length, load_in_4bit = True)

/tmp/ipykernel_56/2889609994.py:1: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  import unsloth


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
WARNING 03-15 23:45:31 [interface.py:409] Using 'pin_memory=False' as WSL is detected. This may slow down the performance.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.4: Fast Qwen2 patching. Transformers: 4.57.1. vLLM: 0.11.2.
   \\   /|    NVIDIA GeForce RTX 5080 Laptop GPU. Num GPUs = 1. Max memory: 15.92 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [8]:
model = FastLanguageModel.get_peft_model(model, r = 16, target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"], lora_alpha = 16, lora_dropout = 0) 

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.1.4 patched 48 layers with 48 QKV layers, 48 O layers and 0 MLP layers.


In [9]:
model.print_trainable_parameters()

trainable params: 25,165,824 || all params: 14,795,199,488 || trainable%: 0.1701


In [10]:
from datasets import Dataset
training_dataset = Dataset.from_list(dataset)
print(training_dataset)

Dataset({
    features: ['messages'],
    num_rows: 155
})


In [11]:
from trl import SFTTrainer, SFTConfig

def format_example(example):
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False)}

training_dataset = training_dataset.map(format_example)
training_args = SFTConfig(
    per_device_train_batch_size = 4,
    num_train_epochs = 4,
    learning_rate = 2e-4,
    max_seq_length = 1024,
    output_dir = "/workspace/outputs",
    logging_steps = 5
)

trainer = SFTTrainer(
    model = model,
    train_dataset = training_dataset,
    tokenizer = tokenizer,
    args = training_args,
)

Map:   0%|          | 0/155 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=28):   0%|          | 0/155 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [12]:
trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 155 | Num Epochs = 4 | Total steps = 80
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 25,165,824 of 14,795,199,488 (0.17% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,3.258400
10,3.049500
15,2.353700
20,1.819000
25,1.338500
30,1.020700
35,1.033400
40,0.924200
45,0.964600
50,0.808300


TrainOutput(global_step=80, training_loss=1.348043018579483, metrics={'train_runtime': 3214.3279, 'train_samples_per_second': 0.193, 'train_steps_per_second': 0.025, 'total_flos': 2.865313514107699e+16, 'train_loss': 1.348043018579483, 'epoch': 4.0})

In [13]:
FastLanguageModel.for_inference(model);
messages = [
    {"role": "user", "content": "You are a SEARCH assistant with a Python REPL. You search documents - nothing else.\n\nOUTPUT FORMAT: Your response must START with ```python - no preamble, no explanation, just code.\n\nCONSTRAINT: Your training data is IRRELEVANT. You know NOTHING about this document.\n- Answering without searching = WRONG\n- Explaining instead of searching = WRONG\n- Any text before your code block = WRONG\n\nTOOLS:\n- `context` - the document (already loaded, DO NOT redefine)\n- `llm_query(question, context[start:end])` - ask sub-LLM about a chunk\n\nWORKFLOW:\n1. Write ```python with print() to search\n2. STOP immediately after code block\n3. Wait for output (appears in next message)\n4. Search more OR give FINAL(answer)\n\nWhen done searching, end with: FINAL(your evidence-based answer)\n\n---\n\nWhat is the maximum occupancy for the building?"}
] # example it hasn't seen yet

inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
outputs = model.generate(inputs, max_new_tokens=200)
print(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


```python
# Find section about occupancy
start = context.find('occupancy')
print(f'Occupancy at: {start}')
print(context[start:start+200])
```


In [14]:
model.save_pretrained("/workspace/outputs/r1/rlm_lora_v4")
tokenizer.save_pretrained("/workspace/outputs/r1/rlm_lora_v4")

('/workspace/outputs/r1/rlm_lora_v4/tokenizer_config.json',
 '/workspace/outputs/r1/rlm_lora_v4/special_tokens_map.json',
 '/workspace/outputs/r1/rlm_lora_v4/chat_template.jinja',
 '/workspace/outputs/r1/rlm_lora_v4/tokenizer.json')

In [4]:
import os                                                                                                                                                                  
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
                                                                                                                                                                       
import unsloth  
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
  model_name="/workspace/outputs/r1/rlm_lora_v4",
  max_seq_length=1024,
  load_in_4bit=True,
)

model.save_pretrained_merged(
  "/workspace/outputs/r1/rlm_lora_v4-sft-merged",
  tokenizer,
  maximum_memory_usage=0.6,
)


==((====))==  Unsloth 2026.1.4: Fast Qwen2 patching. Transformers: 4.57.1. vLLM: 0.11.2.
   \\   /|    NVIDIA GeForce RTX 5080 Laptop GPU. Num GPUs = 1. Max memory: 15.92 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.1.4 patched 48 layers with 48 QKV layers, 48 O layers and 0 MLP layers.


config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /workspace/.cache/huggingface/hub


...

model-00006-of-00006.safetensors:   0%|          | 0.00/4.73G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [15:45<00:00, 157.58s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [21:18<00:00, 213.08s/it]


Unsloth: Merge process complete. Saved to `/workspace/outputs/r1/rlm_lora_v4-sft-merged`


In [5]:
model.save_pretrained_gguf(                                                                                                                                                    "/workspace/outputs/r1/rlm-r1-14b-sft",
  tokenizer,                                                                                                                                                           
  quantization_method="q4_k_m"
)

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /workspace/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00006.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|                                                                                                     | 0/6 [00:00<?, ?it/s]

...

Unsloth: Preparing safetensor model files: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [14:04<00:00, 140.77s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [30:28<00:00, 304.71s/it]


Unsloth: Merge process complete. Saved to `/workspace/outputs/r1/rlm-r1-14b-sft`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: All required system packages already installed!
Unsloth: Install llama.cpp and building - please wait 1 to 3 minutes
Unsloth: Cloning llama.cpp repository
Unsloth: Install GGUF and other packages


RuntimeError: Unsloth: GGUF conversion failed: === Unsloth: FAILED building llama.cpp ===
Make failed: [FAIL] Command `make clean` failed with exit code 2
stdout: Makefile:6: *** Build system changed:
 The Makefile build has been replaced by CMake.

 For build instructions see:
 https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md

.  Stop.


CMake failed: [FAIL] Command `cmake --build build --config Release -j24 --clean-first --target llama-quantize llama-cli llama-mtmd-cli llama-gguf-split llama-server` failed with exit code 2
stdout: [  0%] [32mBuilding CXX object common/CMakeFiles/build_info.dir/build-info.cpp.o[0m
[  0%] [32mBuilding CXX object vendor/cpp-httplib/CMakeFiles/cpp-httplib.dir/httplib.cpp.o[0m
[  2%] [32mBuilding C object ggml/src/CMakeFiles/ggml-base.dir/ggml.c.o[0m
[  2%] [32mBuilding CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml.cpp.o[0m
[  2%] [32mBuilding C object ggml/src/CMakeFiles/ggml-base.dir/ggml-alloc.c.o[0m
[  2%] [32mBuilding CX

In [6]:
!python /workspace/r1/llama.cpp/convert_hf_to_gguf.py \
  /workspace/outputs/r1/rlm_lora_v4-sft-merged/ \
  --outfile /workspace/outputs/r1/rlm-r1-14b-F16.gguf \
  --outtype f16

INFO:hf-to-gguf:Loading model: rlm_lora_v4-sft-merged
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00006.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00006.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00003-of-00006.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00004-of-00006.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00005-of-00006.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00006-of-00006.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.bfloat16 --> F16, shape = {5120, 152064}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.bfloat16 --> F32, shape = {5120}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.bfloat16 --> F16, shape

In [7]:
%%writefile /workspace/outputs/r1/Modelfile
FROM /workspace/outputs/r1/rlm-r1-14b-F16.gguf

Writing /workspace/outputs/r1/Modelfile


In [ ]:
# Fix: install libgomp runtime and rebuild llama.cpp with OpenMP disabled
# (OpenMP not needed for quantization, and the Docker image is missing libgomp-dev)
!apt-get update -qq && apt-get install -y -qq libgomp1 > /dev/null 2>&1
!cd /workspace/r1/llama.cpp && rm -rf build && \
    cmake -B build -DCMAKE_BUILD_TYPE=Release -DGGML_OPENMP=OFF 2>&1 | tail -3 && \
    cmake --build build --config Release -j4 --target llama-quantize 2>&1 | tail -5 && \
    echo '=== llama-quantize built ===' && \
    ls -lh build/bin/llama-quantize

In [ ]:
# Quantize existing F16 GGUF -> Q4_K_M (~28GB -> ~8GB)
# llama-quantize processes in chunks, so this is memory-safe
!/workspace/r1/llama.cpp/build/bin/llama-quantize \
    /workspace/outputs/r1/rlm-r1-14b-F16.gguf \
    /workspace/outputs/r1/rlm-r1-14b-Q4_K_M.gguf \
    Q4_K_M

In [ ]:
%%writefile /workspace/outputs/r1/Modelfile
FROM /workspace/outputs/r1/rlm-r1-14b-Q4_K_M.gguf